# 教程: 结果分析 - 流量历时曲线 (FDC)

## 目的
在水文分析中，仅仅观察流量过程线是不够的。我们需要一些统计工具来更好地理解一条河流的流态特征。**流量历时曲线（Flow Duration Curve, FDC）** 就是这样一种强大的工具。它表示了在一系列时间（通常是长期的日流量数据）中，某流量被达到或超过的频率（或百分比）。

FDC可以告诉我们河流的许多信息，例如：
- **高流量（洪水）特征**: 曲线左侧的形态反映了洪水事件的频率和强度。
- **低流量（枯水）特征**: 曲线右侧的形态反映了河流的基流特性。平坦的右侧尾部通常意味着有稳定的地下水补给。
- **流态的稳定性**: 整条曲线的坡度反映了河流流量的变率。坡度陡峭的曲线意味着河流流量变化剧烈（“暴涨暴落型”），而平缓的曲线则意味着流量稳定。

此笔记本将：
1.  加载一个模拟生成的流量时间序列。
2.  分步演示如何从时间序列数据计算出FDC。
3.  绘制标准的流量历时曲线（通常使用对数坐标轴）。
4.  解释如何解读FDC图。

## 1. 环境设置和加载数据

我们使用`run_example.py`生成的模拟结果作为我们的分析数据。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 加载数据
df = pd.read_csv('../examples/results/simulation_results.csv', index_col='date', parse_dates=True)

# 我们只分析总出口的流量 (simulated_flow_1)
flows = df['simulated_flow_1'].dropna() # 确保没有缺失值

print("成功加载流量数据。数据点总数:", len(flows))

## 2. 计算流量历时曲线 (FDC)

计算FDC的步骤如下：
1.  **排序**: 将所有的流量数据从大到小进行排序。
2.  **排名**: 为每个排好序的流量值分配一个排名（Rank），从1开始到N（N是数据点的总数）。
3.  **计算超越概率**: 对每个流量值，计算其被达到或超过的概率 `P`。常用的公式是 `P = Rank / (N + 1)`。乘以100可以得到超越频率（%）。

In [ ]:
# 1. 排序
sorted_flows = np.sort(flows)[::-1] # [::-1] 实现降序

# 2. 排名
n = len(sorted_flows)
ranks = np.arange(1, n + 1)

# 3. 计算超越概率
exceedance_prob = (ranks / (n + 1)) * 100

print("FDC计算完成。")
print("\n前5个流量值及其超越概率:")
for i in range(5):
    print(f"  Flow: {sorted_flows[i]:.2f} m³/s, Exceedance: {exceedance_prob[i]:.2f}%")

## 3. 绘制FDC图

我们以超越概率为X轴，流量为Y轴，来绘制FDC。为了更好地展示流量在多个数量级上的变化，Y轴通常采用对数坐标。

In [ ]:
plt.figure(figsize=(10, 7))

plt.plot(exceedance_prob, sorted_flows)

# 设置对数坐标轴
plt.yscale('log')

plt.title('Flow Duration Curve (FDC)')
plt.xlabel('Exceedance Probability (%)')
plt.ylabel('Discharge (m³/s) [log scale]')
plt.grid(True, which='both') # 'both' 会为主、次刻度都画网格线
plt.show()

### 如何解读FDC图

- **Q5 (高流量)**: 在X轴上找到5%，对应的Y值就是有5%的时间被达到或超过的流量。这通常代表了高流量或洪水的情况。
- **Q50 (中位流量)**: 对应的Y值是中位流量，它有一半的时间被超过，一半的时间未被超过。
- **Q95 (低流量)**: 对应的Y值是有95%的时间被超过的流量，这代表了河流的枯水或基流情况。

这条FDC曲线的坡度比较陡峭，说明这条（模拟的）河流流量变率很大，对降雨的响应很强，而基流部分则相对较小。这与我们运行的事件驱动的、基于SCS产流模型的模拟结果是相符的。